# Phase 4: Feature Engineering & Model Building
This notebook covers the model training pipeline for the AI-Powered Job Fraud Detection project. We vectorize the cleaned text using TF-IDF, address class imbalance with balanced weights, train a Logistic Regression classifier, evaluate its performance, and export the trained model artifacts.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
import joblib

In [2]:
# Load cleaned dataset
df = pd.read_csv('cleaned_jobs.csv')
df['text'] = df['text'].fillna('')

print(f"Dataset shape: {df.shape}")
print(f"Genuine postings: {sum(df['fraudulent'] == 0)}")
print(f"Fraudulent postings: {sum(df['fraudulent'] == 1)}")

Dataset shape: (17880, 19)
Genuine postings: 17014
Fraudulent postings: 866


## 1. Train-Test Split

In [3]:
X = df['text']
y = df['fraudulent']

# 80% train, 20% test split, stratified to preserve class ratios
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 2. Text Vectorization using TF-IDF

In [4]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF shape (Train): {X_train_tfidf.shape}")

TF-IDF shape (Train): (14304, 5000)


## 3. Model Training (Logistic Regression with Class Balancing)

In [5]:
# We use class_weight='balanced' to automatically adjust weights inversely proportional to class frequencies
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

## 4. Evaluation

In [6]:
y_pred = model.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9692

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      3403
           1       0.63      0.90      0.74       173

    accuracy                           0.97      3576
   macro avg       0.81      0.93      0.86      3576
weighted avg       0.98      0.97      0.97      3576



In [7]:
# Visualize Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Genuine', 'Fraudulent'])
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

## 5. Exporting Model & Vectorizer

In [8]:
os.makedirs('model', exist_ok=True)
joblib.dump(model, 'model/fraud_detector.pkl')
joblib.dump(vectorizer, 'model/tfidf_vectorizer.pkl')
print("Model artifacts saved successfully!")

Model artifacts saved successfully!
